# Guía de Adquisición, Preparación y Análisis Exploratorio de Datos

Este cuaderno utiliza el dataset **`intel-cpu-dataset`** para demostrar las etapas del flujo de datos orientado al análisis de métricas operativas de infraestructura de servidores.

**Contenido:**
1. Adquisición de datos
2. Información básica del dataset
3. Limpieza de datos — comparación de estrategias
4. Análisis exploratorio (EDA) sobre datos imputados
5. Visualización avanzada
6. Conclusiones

---
**Universidad ECCI · Electiva II — DevOps · 2026**  
**Autores:** Julian David Garzon Medina · Javier Stiven Amaya Devia


## 1. Adquisición de datos

El dataset `intel-cpu-dataset` contiene registros de métricas operativas de máquinas virtuales bajo distintas condiciones de carga de trabajo.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

sns.set(style="whitegrid")
plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white'})

print("✅ Librerías importadas correctamente")


> **Nota:** Si tienes el archivo en tu repositorio usa la **Opción A**.  
> Si necesitas subirlo manualmente en Colab usa la **Opción B**.


In [ ]:
# ── Opción A: desde URL del repositorio ─────────────────────────────
# url = 'https://raw.githubusercontent.com/tu_usuario/tu_repo/main/data/intel_dataset.csv'
# df = pd.read_csv(url)

# ── Opción B: subir manualmente en Colab ────────────────────────────
from google.colab import files
uploaded = files.upload()

import io
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print("Primeras 5 filas del DataFrame:")
df.head()


## 2. Información básica del dataset

Antes de comenzar cualquier limpieza es recomendable entender la estructura del DataFrame:
- **Número de filas y columnas** (`shape`)
- **Tipos de datos** y valores no nulos (`info`)
- **Resumen estadístico** de variables numéricas (`describe`)


In [ ]:
dimension = df.shape
print(f"Dimensiones (filas, columnas): {dimension}")
print()
df.info()


In [ ]:
print("Resumen estadístico de variables numéricas:")
df.describe(include=[np.number]).round(3)


In [ ]:
NUM_COLS = [
    'cpu_usage', 'memory_usage', 'network_traffic',
    'power_consumption', 'num_executed_instructions',
    'execution_time', 'energy_efficiency'
]
CAT_COLS = ['task_type', 'task_priority', 'task_status']
ID_COLS  = ['vm_id', 'timestamp']

print(f"Variables NUMÉRICAS   ({len(NUM_COLS)}): {NUM_COLS}")
print(f"Variables CATEGÓRICAS ({len(CAT_COLS)}): {CAT_COLS}")
print(f"Variables ID          ({len(ID_COLS)}):  {ID_COLS}")


## 3. Limpieza de datos — comparación de estrategias

Una vez inspeccionado el dataset, se procede a la limpieza. Se evalúan tres estrategias y se compara su impacto estadístico para seleccionar la más adecuada.


In [ ]:
# Valores nulos por columna
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_pct    = (df.isnull().sum() / len(df) * 100).round(2).sort_values(ascending=False)

missing_df = pd.DataFrame({'Nulos': missing_values, 'Porcentaje (%)': missing_pct})
print("Valores nulos por columna:")
print(missing_df.to_string())
print()
print(f"Filas con AL MENOS 1 nulo:   {df.isnull().any(axis=1).sum():,} ({df.isnull().any(axis=1).mean()*100:.1f}%)")
print(f"Filas completamente limpias: {df.dropna().shape[0]:,} ({df.dropna().shape[0]/len(df)*100:.1f}%)")


Las columnas numéricas clave presentan entre **9.8% y 10.8%** de valores nulos. Dependiendo la proporción y relevancia de la variable se pueden tomar varias decisiones:

1. **Eliminar filas** con valores ausentes.
2. **Imputación con la mediana** — robusta ante valores extremos.
3. **Imputación con la media** — sensible a la distribución de los datos.

A continuación se aplican las tres y se compara su impacto estadístico.


### 3.1 Análisis del patrón de nulos


In [ ]:
null_por_fila = df.isnull().sum(axis=1)

print("Distribución de nulos por fila:")
print(null_por_fila.value_counts().sort_index().to_string())
print()
print(f"Filas con 8 o más columnas nulas: {(null_por_fila >= 8).sum()} "
      f"({(null_por_fila >= 8).mean()*100:.1f}% del total)")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

missing_sorted = missing_pct.sort_values(ascending=True)
colors_bar = ['#E74C3C' if v > 10 else '#2E75B6' for v in missing_sorted]
axes[0].barh(missing_sorted.index, missing_sorted.values,
             color=colors_bar, alpha=0.85, edgecolor='white')
axes[0].axvline(10, color='#E74C3C', linestyle='--', linewidth=1.5, label='Umbral 10%')
axes[0].set_xlabel('% Valores Faltantes')
axes[0].set_title('Porcentaje de nulos por variable', fontweight='bold')
axes[0].legend()
for i, v in enumerate(missing_sorted):
    axes[0].text(v + 0.1, i, f'{v:.1f}%', va='center', fontsize=8)

sample_null = df.sample(min(250, len(df)), random_state=42).isnull().astype(int)
sns.heatmap(sample_null.T, cmap=['#EBF3FB', '#2E75B6'], cbar=False,
            ax=axes[1], linewidths=0, xticklabels=False)
axes[1].set_title('Patrón de nulos — muestra 250 filas
(azul = valor nulo)', fontweight='bold')
axes[1].tick_params(axis='y', labelsize=8)

plt.suptitle('Análisis de Valores Faltantes', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


### 3.2 Método 1: Eliminar filas con valores ausentes


In [ ]:
df_drop = df.dropna(subset=NUM_COLS).copy()

print(f"Filas originales:                {df.shape[0]:,}")
print(f"Filas después de eliminar nulos: {df_drop.shape[0]:,}")
print(f"Filas eliminadas:                {df.shape[0] - df_drop.shape[0]:,} "
      f"({(df.shape[0]-df_drop.shape[0])/df.shape[0]*100:.1f}%)")


### 3.3 Método 2: Imputación con la mediana

La mediana es robusta ante distribuciones asimétricas y valores extremos. Se reemplaza cada valor faltante con la mediana calculada sobre los valores existentes de esa columna.


In [ ]:
df_median = df.copy()

numeric_imputer  = SimpleImputer(strategy='median')
category_imputer = SimpleImputer(strategy='most_frequent')

df_median[NUM_COLS] = numeric_imputer.fit_transform(df_median[NUM_COLS])
df_median[CAT_COLS] = category_imputer.fit_transform(df_median[CAT_COLS])

print("Nulos después de imputación con mediana:")
print(df_median[NUM_COLS + CAT_COLS].isnull().sum().to_string())
print()
print(f"Registros conservados: {len(df_median):,} (100% del dataset original)")
df_median[NUM_COLS].head()


### 3.4 Método 3: Imputación con la media

La media es sensible a valores extremos y a la distribución de los datos. En distribuciones simétricas como la de este dataset, media y mediana son muy similares, por lo que el impacto de elegir una u otra es mínimo.


In [ ]:
df_mean = df.copy()

mean_imputer = SimpleImputer(strategy='mean')

df_mean[NUM_COLS] = mean_imputer.fit_transform(df_mean[NUM_COLS])
df_mean[CAT_COLS] = category_imputer.fit_transform(df_mean[CAT_COLS])

print("Nulos después de imputación con media:")
print(df_mean[NUM_COLS + CAT_COLS].isnull().sum().to_string())
print()
print(f"Registros conservados: {len(df_mean):,} (100% del dataset original)")
df_mean[NUM_COLS].head()


### 3.5 Comparación estadística entre estrategias


In [ ]:
# Tabla comparativa de medias
comparison_mean = pd.DataFrame({
    'Original (con nulos)': df[NUM_COLS].mean(),
    'Eliminar filas':       df_drop[NUM_COLS].mean(),
    'Imputación mediana':   df_median[NUM_COLS].mean(),
    'Imputación media':     df_mean[NUM_COLS].mean(),
}).round(4)

print("Comparación de MEDIAS por estrategia:")
display(comparison_mean)


In [ ]:
# Tabla comparativa de desviaciones estándar
comparison_std = pd.DataFrame({
    'Original (con nulos)': df[NUM_COLS].std(),
    'Eliminar filas':       df_drop[NUM_COLS].std(),
    'Imputación mediana':   df_median[NUM_COLS].std(),
    'Imputación media':     df_mean[NUM_COLS].std(),
}).round(4)

print("Comparación de DESVIACIÓN ESTÁNDAR por estrategia:")
display(comparison_std)

print()
print("💡 La imputación reduce la desviación estándar (~5-6% menor que el original).")
print("   Esto ocurre porque los valores imputados (mediana/media) son centrales")
print("   y reducen artificialmente la variabilidad del dataset.")


In [ ]:
# Visualización comparativa: distribuciones antes y después de imputar
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('Impacto de la imputación sobre las distribuciones
'
             '(azul = eliminar filas  |  naranja = mediana  |  verde = media)',
             fontsize=13, fontweight='bold', y=1.01)

for i, col in enumerate(NUM_COLS):
    ax = axes[i // 4][i % 4]
    data_drop   = df_drop[col].dropna()
    data_median = df_median[col]
    data_mean   = df_mean[col]

    ax.hist(data_drop,   bins=35, alpha=0.45, color='#2E75B6',
            density=True, label='Eliminar filas', edgecolor='white', linewidth=0.2)
    ax.hist(data_median, bins=35, alpha=0.45, color='#E67E22',
            density=True, label='Mediana',        edgecolor='white', linewidth=0.2)
    ax.hist(data_mean,   bins=35, alpha=0.35, color='#27AE60',
            density=True, label='Media',          edgecolor='white', linewidth=0.2)

    ax.set_title(col.replace('_', ' ').title(), fontweight='bold', fontsize=9)
    ax.set_ylabel('Densidad', fontsize=8)
    if i == 0:
        ax.legend(fontsize=7, framealpha=0.8)

axes[1][3].set_visible(False)
plt.tight_layout()
plt.show()

print("💡 Las distribuciones de los tres métodos son visualmente muy similares.")
print("   La imputación no distorsiona la forma general de las distribuciones.")


In [ ]:
# Comparación de skewness y kurtosis
comp_shape = pd.DataFrame({
    'Original':          df[NUM_COLS].skew(),
    'Eliminar filas':    df_drop[NUM_COLS].skew(),
    'Imputación mediana':df_median[NUM_COLS].skew(),
    'Imputación media':  df_mean[NUM_COLS].skew(),
}).round(4)

print("Comparación de ASIMETRÍA (skewness) por estrategia:")
display(comp_shape)

comp_kurt = pd.DataFrame({
    'Original':          df[NUM_COLS].kurtosis(),
    'Eliminar filas':    df_drop[NUM_COLS].kurtosis(),
    'Imputación mediana':df_median[NUM_COLS].kurtosis(),
    'Imputación media':  df_mean[NUM_COLS].kurtosis(),
}).round(4)

print()
print("Comparación de CURTOSIS por estrategia:")
display(comp_kurt)

print()
print("💡 La imputación eleva levemente la curtosis (aprox. +0.2) respecto al original.")
print("   Esto indica que la distribución se vuelve marginalmente menos plana en el centro,")
print("   ya que los valores imputados se concentran en la mediana/media.")


> **Estrategia seleccionada: imputación con la mediana.**  
> Dado que las distribuciones del dataset son quasi-uniformes y simétricas,  
> la mediana y la media producen resultados prácticamente idénticos.  
> Se elige la mediana por ser más robusta ante posibles valores extremos  
> en datos de producción futuros. El DataFrame de trabajo será `df_clean`.


In [ ]:
df_clean = df_median.copy()
print(f"✅ DataFrame de trabajo: {df_clean.shape[0]:,} filas × {df_clean.shape[1]} columnas")
df_clean[NUM_COLS].describe().round(3)


## 4. Análisis Exploratorio de Datos (EDA)

Una vez aplicada la imputación, realizamos el análisis exploratorio completo sobre `df_clean` (2.081 registros). Se incluye estadística descriptiva, distribuciones, detección de valores atípicos y análisis de correlaciones.


### 4.1 Estadística descriptiva


In [ ]:
desc = df_clean[NUM_COLS].describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95])
print("Estadística descriptiva completa (datos imputados — mediana):")
display(desc.round(3))


In [ ]:
print(f"{'Variable':<35} {'Skewness':>10} {'Kurtosis':>10}  Interpretación")
print('-' * 78)
for col in NUM_COLS:
    sk = df_clean[col].skew()
    ku = df_clean[col].kurtosis()
    interp = 'Quasi-uniforme (platicúrtica)' if abs(ku) > 1 else 'Levemente menos plana por imputación'
    print(f"{col:<35} {sk:>+10.4f} {ku:>10.4f}  {interp}")

print()
print("💡 Skewness se mantiene cercano a 0 → la imputación no introdujo sesgo.")
print("   Kurtosis sube levemente respecto al original (~-1.2 → ~-1.0)")
print("   por concentración de valores imputados en la mediana central.")


### 4.2 Distribuciones


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('Distribución de Variables Numéricas — datos imputados (mediana)
'
             '(rojo = media  |  verde = mediana)',
             fontsize=13, fontweight='bold', y=1.01)

colors = ['#2E75B6','#1F3864','#27AE60','#E67E22','#8E44AD','#E74C3C','#16A085']

for i, (col, color) in enumerate(zip(NUM_COLS, colors)):
    ax = axes[i // 4][i % 4]
    data = df_clean[col]
    sns.histplot(data, bins=35, kde=True, ax=ax, color=color, alpha=0.6)
    ax.axvline(data.mean(),   color='#E74C3C', linewidth=1.8, linestyle='--',
               label=f'Media={data.mean():.1f}')
    ax.axvline(data.median(), color='#27AE60', linewidth=1.8, linestyle=':',
               label=f'Mediana={data.median():.1f}')
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
    ax.set_ylabel('Frecuencia')
    ax.legend(fontsize=7.5, framealpha=0.7)

axes[1][3].set_visible(False)
plt.tight_layout()
plt.show()

print("💡 Se observa un pequeño pico en el centro de cada distribución.")
print("   Es el efecto de la imputación: los ~200 valores nulos de cada variable")
print("   se reemplazaron con su mediana, creando una concentración artificial en ese punto.")


> **Observación clave:** en las distribuciones del dataset imputado es visible un pequeño pico en el centro de cada histograma. Este es el **efecto de la imputación**: aproximadamente 200 valores nulos por variable fueron reemplazados por la mediana, generando una concentración artificial en ese punto. Este comportamiento es esperable y documentado en la literatura de preprocesamiento de datos.


### 4.3 Valores atípicos


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle('Boxplots — Detección de Valores Atípicos (datos imputados)', fontsize=13, fontweight='bold')

for i, (col, color) in enumerate(zip(NUM_COLS, colors)):
    ax = axes[i // 4][i % 4]
    sns.boxplot(y=df_clean[col], ax=ax, color=color, width=0.5,
                flierprops=dict(marker='o', markerfacecolor='#E74C3C',
                                markersize=4, alpha=0.6))
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold')

axes[1][3].set_visible(False)
plt.tight_layout()
plt.show()

print(f"{'Variable':<35} {'Q1':>8} {'Q3':>8} {'Lím.inf':>10} {'Lím.sup':>10} {'Outliers':>10}")
print('-' * 85)
for col in NUM_COLS:
    q1, q3 = df_clean[col].quantile([0.25, 0.75])
    iqr    = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    n      = ((df_clean[col] < lo) | (df_clean[col] > hi)).sum()
    print(f"{col:<35} {q1:>8.2f} {q3:>8.2f} {lo:>10.2f} {hi:>10.2f} {n:>8,} ({n/len(df_clean)*100:.1f}%)")

print()
print("💡 Cero outliers univariados también en el dataset imputado.")
print("   La imputación con mediana no introduce valores atípicos artificiales.")


### 4.4 Correlaciones


In [ ]:
corr_matrix = df_clean[NUM_COLS].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.3f',
            linewidths=0.5, square=True,
            xticklabels=[c.replace('_usage','').replace('_','
') for c in NUM_COLS],
            yticklabels=[c.replace('_usage','').replace('_','
') for c in NUM_COLS])
plt.title('Matriz de correlación de Pearson — datos imputados',
          fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print("Correlaciones más altas en valor absoluto:")
corr_pairs = (corr_matrix.abs()
              .unstack()
              .sort_values(ascending=False)
              .drop_duplicates())
corr_pairs = corr_pairs[corr_pairs < 1.0]
print(corr_pairs.head(8).to_string())
print()
print("💡 Correlaciones |r| < 0.05 en todos los pares → variables independientes.")
print("   La imputación no introdujo correlaciones artificiales entre variables.")


## 5. Visualización Avanzada

Exploración de relaciones entre múltiples variables y comportamiento de las variables categóricas sobre el dataset imputado completo.


In [ ]:
# Variables categóricas
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Distribución de Variables Categóricas', fontsize=13, fontweight='bold')

cat_info = [
    ('task_type',     'Tipo de Tarea',      ['#1F3864','#2E75B6','#9DC3E6']),
    ('task_priority', 'Prioridad de Tarea', ['#27AE60','#F39C12','#E74C3C']),
    ('task_status',   'Estado de Tarea',    ['#8E44AD','#2980B9','#27AE60']),
]

for ax, (col, title, pal) in zip(axes, cat_info):
    vc    = df_clean[col].value_counts(dropna=True)
    bars  = ax.bar(vc.index, vc.values, color=pal[:len(vc)], alpha=0.85, edgecolor='white')
    total = vc.sum()
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Registros')
    ax.set_ylim(0, vc.max() * 1.2)
    for bar, val in zip(bars, vc.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 5,
                f'{int(val)}
({val/total*100:.1f}%)',
                ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("Distribución balanceada en todas las categorías (~33% por clase).")


In [ ]:
# Boxplot por tipo de tarea
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribución de métricas según tipo de tarea', fontsize=13, fontweight='bold')

sns.boxplot(x='task_type', y='cpu_usage',    data=df_clean,
            palette='Blues',  ax=axes[0], hue='task_type', legend=False)
axes[0].set_title('CPU Usage por tipo de tarea', fontweight='bold')
axes[0].set_xlabel('Tipo de tarea')
axes[0].set_ylabel('CPU Usage (%)')

sns.boxplot(x='task_type', y='memory_usage', data=df_clean,
            palette='Greens', ax=axes[1], hue='task_type', legend=False)
axes[1].set_title('Memory Usage por tipo de tarea', fontweight='bold')
axes[1].set_xlabel('Tipo de tarea')
axes[1].set_ylabel('Memory Usage (%)')

plt.tight_layout()
plt.show()

print("Estadísticas por tipo de tarea:")
display(df_clean.groupby('task_type')[NUM_COLS[:4]].agg(['mean','std']).round(2))


In [ ]:
# Scatter: CPU vs Memory coloreado por tipo de tarea
plt.figure(figsize=(9, 6))
palette = {'io': '#1F3864', 'compute': '#2E75B6', 'network': '#9DC3E6'}
for task, grp in df_clean.groupby('task_type'):
    plt.scatter(grp['cpu_usage'], grp['memory_usage'],
                label=task, alpha=0.3, s=10, color=palette.get(task, 'gray'))
plt.xlabel('CPU Usage (%)')
plt.ylabel('Memory Usage (%)')
plt.title('CPU vs Memory — por tipo de tarea (datos imputados)', fontweight='bold')
plt.legend(title='task_type')
plt.tight_layout()
plt.show()

print("Las tres categorías se distribuyen de forma casi idéntica en el espacio CPU/Memory.")


In [ ]:
# Scatter matrix de 4 variables principales
sample_pairs = df_clean[['cpu_usage','memory_usage',
                          'network_traffic','power_consumption']]               .sample(min(300, len(df_clean)), random_state=42)
sample_pairs.columns = ['CPU','Memory','Network','Power']

g = sns.pairplot(sample_pairs, diag_kind='hist',
                 plot_kws=dict(alpha=0.3, s=8, color='#2E75B6'),
                 diag_kws=dict(color='#2E75B6', alpha=0.7))
g.figure.suptitle('Parejas de variables numéricas (muestra 300 registros — datos imputados)',
                   y=1.01, fontsize=12, fontweight='bold')

for i in range(4):
    for j in range(4):
        if i != j:
            r, _ = stats.pearsonr(sample_pairs.iloc[:, i], sample_pairs.iloc[:, j])
            g.axes[i][j].annotate(f'r={r:.3f}', xy=(0.05, 0.90),
                                  xycoords='axes fraction',
                                  fontsize=8, color='#E74C3C')
plt.show()


In [ ]:
# Boxplot por estado de tarea
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribución de métricas según estado de tarea', fontsize=13, fontweight='bold')

sns.boxplot(x='task_status', y='cpu_usage', data=df_clean,
            palette='Oranges', ax=axes[0], hue='task_status', legend=False)
axes[0].set_title('CPU Usage por estado de tarea', fontweight='bold')
axes[0].set_xlabel('Estado')
axes[0].set_ylabel('CPU Usage (%)')

sns.boxplot(x='task_status', y='network_traffic', data=df_clean,
            palette='Purples', ax=axes[1], hue='task_status', legend=False)
axes[1].set_title('Network Traffic por estado de tarea', fontweight='bold')
axes[1].set_xlabel('Estado')
axes[1].set_ylabel('Network Traffic')

plt.tight_layout()
plt.show()

print("Estadísticas por estado de tarea:")
display(df_clean.groupby('task_status')[['cpu_usage','memory_usage','network_traffic']]        .agg(['mean','std']).round(2))


## 6. Conclusiones

En este cuaderno se realizó el ciclo completo de adquisición, limpieza y análisis exploratorio del dataset `intel-cpu-dataset`.

- **Adquisición:** 2.081 registros, 12 variables (7 numéricas, 3 categóricas, 2 de identificación).
- **Limpieza:** se compararon tres estrategias. Se seleccionó la **imputación con la mediana**, que conserva los 2.081 registros originales y no introduce sesgo ni correlaciones artificiales.
- **EDA:** las distribuciones del dataset imputado son quasi-uniformes y simétricas, con un pequeño pico central visible producto de la imputación. Las correlaciones se mantienen por debajo de |r| < 0.05 en todos los pares. No se detectaron outliers univariados.
- **Visualización avanzada:** las variables categóricas presentan distribución balanceada. Los boxplots por tipo de tarea y estado confirman que no existe concentración diferenciada según la categoría de carga.

### Hallazgos clave

| Hallazgo | Detalle |
|---|---|
| Imputación no introduce sesgo | Skewness se mantiene ≈ 0 antes y después |
| Pequeña reducción de variabilidad | σ cae ~5-6% respecto al original por efecto de los valores centrales imputados |
| Pico central en histogramas | Efecto esperado y documentado de la imputación con estadístico central |
| Correlaciones se mantienen nulas | |r| < 0.05 — la imputación no crea relaciones artificiales |
| Cero outliers univariados | Se mantiene igual que en el dataset con filas eliminadas |
| Categóricas balanceadas | ~33% por categoría en las tres variables |


In [ ]:
print('=' * 60)
print('RESUMEN EDA — intel-cpu-dataset (imputación mediana)')
print('=' * 60)
print(f"  Registros totales:              {len(df_clean):,}")
print(f"  Variables numéricas:            {len(NUM_COLS)}")
print(f"  Variables categóricas:          {len(CAT_COLS)}")
print(f"  Skewness promedio:              {df_clean[NUM_COLS].skew().mean():.4f}")
print(f"  Kurtosis promedio:              {df_clean[NUM_COLS].kurtosis().mean():.4f}")
print(f"  Correlación máx. |r|:           "
      f"{df_clean[NUM_COLS].corr().abs().values[np.triu_indices(7,1)].max():.4f}")
print(f"  Outliers univariados (IQR):     0")
print()
print("✅ Dataset analizado y preparado para etapas posteriores del proyecto.")
